# ELIA Engine — Phase 6: Garden Layout Model Training & Export

This Google Colab notebook provides the reproducible training workflow for the **Component 4 (ELIA Engine)** neural layout model.

### Pipeline Overview
1. **Input Tensor ($9 \times 96 \times 96$)**:
   - Ch 0: Land boundary mask
   - Ch 1: House footprint mask
   - Ch 2: Road access corridor
   - Ch 3: Main entrance location
   - Ch 4: Utility safety buffers (well, septic tank)
   - Ch 5: Solar / northern aspect gradient
   - Ch 6: Client greenery density weight (0.33, 0.66, 1.0)
   - Ch 7: Concept style Greeny flag
   - Ch 8: Concept style Minimalist flag

2. **Output Heatmaps ($4 \times 96 \times 96$)**:
   - Ch 0: Tree density map
   - Ch 1: Outdoor seating / dining density map
   - Ch 2: Garden / pathway lighting density map
   - Ch 3: Circulation guidance corridor

3. **Dataset Split**:
   - 237 training records (44 sites)
   - 46 validation records (10 sites)
   - 49 test records (10 sites)
   - Best model selected at **Epoch 17**

4. **Production Export**:
   - PyTorch `.pt` checkpoint
   - Standalone zero-dependency `weights.npz` with SHA-256 manifest for production FastAPI backend inference.

In [ ]:
# 1. Setup & Environment Verification
!pip install -q shapely numpy matplotlib
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Optional: Mount Google Drive for Persistent Storage
# from google.colab import drive
# drive.mount('/content/drive')
# ARTIFACT_DIR = '/content/drive/MyDrive/elia_model_artifacts/garden_layout_v1'
ARTIFACT_DIR = './model_artifacts/garden_layout_v1'
import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)

In [ ]:
# 3. Run PyTorch Training (17 Epochs)
!python -m pip install -q requests
!python training/elia/train_garden_layout.py --output_dir {ARTIFACT_DIR} --epochs 17 --seed 42

In [ ]:
# 4. Verify Export Package Checksum and Load in Fresh Process
import json, hashlib
with open(f"{ARTIFACT_DIR}/manifest.json", "r") as f:
    manifest = json.load(f)
print("Model Manifest:")
print(json.dumps(manifest, indent=2))

# Verify sha256 checksum
sha256 = hashlib.sha256()
with open(f"{ARTIFACT_DIR}/weights.npz", "rb") as f:
    while chunk := f.read(8192):
        sha256.update(chunk)
print(f"Checksum Validated: {sha256.hexdigest() == manifest['weights_sha256']}")